# Exploration — Dataset Mart Jürisoo + Calcul ELO
Pipeline : chargement → nettoyage → calcul ELO chronologique → visualisation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 130
plt.rcParams['font.family'] = 'sans-serif'

## 1. Chargement des données

In [ ]:
# Chemin relatif depuis ml/notebooks/ → remonte deux niveaux vers ml/data/
df = pd.read_csv('../../ml/data/results.csv', parse_dates=['date'])

print(f'Matchs chargés : {len(df):,}')
print(f'Période        : {df["date"].min().date()} → {df["date"].max().date()}')
print(f'Équipes uniques: {pd.concat([df["home_team"], df["away_team"]]).nunique()}')
df.head()

In [ ]:
# Aperçu des types de tournois présents dans le dataset
print(df['tournament'].value_counts().head(20).to_string())

## 2. Calcul des ELO ratings

On parcourt tous les matchs **dans l'ordre chronologique** (obligatoire — l'ELO d'un match dépend de tous les matchs précédents).

Formule :
```
expected_home = 1 / (1 + 10^((elo_away - elo_home) / 400))
score         = 1 (victoire dom.) | 0.5 (nul) | 0 (défaite dom.)
Δ_home        = K × (score − expected_home)
Δ_away        = K × ((1−score) − (1−expected_home))
```

Facteur K par type de compétition :
- **40** → tournoi majeur (Coupe du Monde, Coupe continentale)
- **30** → match de qualification
- **20** → match amical ou autre

In [ ]:
# --- Définition du facteur K ---
# Mots-clés identifiant les tournois majeurs et les qualifications
MAJOR_KEYWORDS = [
    'FIFA World Cup',
    'UEFA Euro',
    'Copa América',
    'Africa Cup of Nations',
    'Asian Cup',
    'Gold Cup',
    'AFC Asian Cup',
    'CONCACAF Gold Cup',
    'Oceania Nations Cup',
    'African Cup of Nations',
]
QUAL_KEYWORDS = ['qualification', 'Qualification', 'qualifier', 'Qualifier']

def get_k(tournament: str) -> int:
    """Retourne le facteur K selon le type de compétition."""
    # Un match de qualification pour la CdM reste K=30, même si 'World Cup' est dans le nom
    is_qual = any(kw in tournament for kw in QUAL_KEYWORDS)
    if is_qual:
        return 30
    is_major = any(kw in tournament for kw in MAJOR_KEYWORDS)
    if is_major:
        return 40
    return 20  # Amical, Nations League, tournoi régional mineur, etc.

# Vérification rapide sur quelques tournois représentatifs
tests = [
    'FIFA World Cup',
    'FIFA World Cup qualification',
    'UEFA Euro',
    'UEFA Euro qualification',
    'Friendly',
    'Copa América',
    'UEFA Nations League',
]
for t in tests:
    print(f'K={get_k(t):2d}  →  {t}')

In [ ]:
# --- Tri chronologique (obligatoire avant tout calcul ELO) ---
df = df.sort_values('date').reset_index(drop=True)

# ELO initial : toutes les équipes démarrent à 1500
elo_current = {}   # {team_name: elo_float}  — état courant de chaque équipe
ELO_INIT = 1500.0

# elo_history contiendra une ligne par (match, équipe) pour tracer l'évolution
history_rows = []  # liste de dicts, convertie en DataFrame à la fin (plus rapide qu'append)

for _, row in df.iterrows():
    home = row['home_team']
    away = row['away_team']
    date = row['date']

    # Récupérer l'ELO actuel (1500 si équipe jamais vue)
    elo_h = elo_current.get(home, ELO_INIT)
    elo_a = elo_current.get(away, ELO_INIT)

    # Probabilité de victoire de l'équipe domicile selon l'ELO
    expected_home = 1.0 / (1.0 + 10.0 ** ((elo_a - elo_h) / 400.0))

    # Score réel : 1 = victoire dom., 0.5 = nul, 0 = défaite dom.
    if row['home_score'] > row['away_score']:
        score = 1.0
    elif row['home_score'] == row['away_score']:
        score = 0.5
    else:
        score = 0.0

    # Facteur K selon le tournoi
    K = get_k(row['tournament'])

    # Mise à jour des ELO
    new_elo_h = elo_h + K * (score - expected_home)
    new_elo_a = elo_a + K * ((1.0 - score) - (1.0 - expected_home))

    # Sauvegarde de l'état après ce match
    elo_current[home] = new_elo_h
    elo_current[away] = new_elo_a

    # Enregistrement dans l'historique (ELO après le match)
    history_rows.append({'date': date, 'team': home, 'elo': new_elo_h})
    history_rows.append({'date': date, 'team': away, 'elo': new_elo_a})

elo_history = pd.DataFrame(history_rows)
print(f'Lignes dans elo_history : {len(elo_history):,}')
elo_history.head(6)

In [ ]:
# --- Classement ELO actuel (fin du dataset) ---
# On prend le dernier ELO connu de chaque équipe
elo_latest = (
    elo_history
    .sort_values('date')
    .groupby('team', as_index=False)
    .last()
    .sort_values('elo', ascending=False)
    .reset_index(drop=True)
)
elo_latest.index += 1  # classement commence à 1
print('Top 20 ELO actuels :')
print(elo_latest[['team', 'elo']].head(20).to_string())

## 3. Évolution ELO des 6 grandes nations (1990 → aujourd'hui)

In [ ]:
NATIONS = ['France', 'Brazil', 'Germany', 'Spain', 'Argentina', 'England']
COLORS  = ['#002395', '#009C3B', '#000000', '#AA151B', '#74ACDF', '#CF081F']
LABELS  = ['France', 'Brésil', 'Allemagne', 'Espagne', 'Argentine', 'Angleterre']

# Filtrer l'historique à partir de 1990 pour les 6 nations
mask = (
    elo_history['team'].isin(NATIONS) &
    (elo_history['date'] >= '1990-01-01')
)
elo_6 = elo_history[mask].copy()

fig, ax = plt.subplots(figsize=(14, 6))

for nation, color, label in zip(NATIONS, COLORS, LABELS):
    data = elo_6[elo_6['team'] == nation].sort_values('date')
    ax.plot(data['date'], data['elo'], label=label, color=color, linewidth=1.6, alpha=0.9)

# Repères visuels : années des Coupes du Monde depuis 1990
world_cups = [1990, 1994, 1998, 2002, 2006, 2010, 2014, 2018, 2022]
for yr in world_cups:
    ax.axvline(pd.Timestamp(f'{yr}-06-01'), color='gray', linestyle='--', linewidth=0.6, alpha=0.5)
    ax.text(pd.Timestamp(f'{yr}-06-01'), ax.get_ylim()[0] + 5, str(yr),
            fontsize=7, color='gray', ha='center')

ax.set_title('Évolution ELO — 6 grandes nations (1990 → présent)', fontsize=14, fontweight='bold')
ax.set_xlabel('Année')
ax.set_ylabel('ELO rating')
ax.xaxis.set_major_locator(mdates.YearLocator(4))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.legend(loc='upper left', framealpha=0.9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../elo_evolution.png', bbox_inches='tight')
plt.show()
print('Graphique sauvegardé → ml/elo_evolution.png')

## 4. Vérification des données — matchs Coupe du Monde

In [ ]:
# Isoler les matchs FIFA World Cup (phase finale uniquement, hors qualifications)
wc = df[df['tournament'] == 'FIFA World Cup'].copy()

print(f'Matchs World Cup dans le dataset : {len(wc)}')
print(f'Editions couvertes              : {sorted(wc["date"].dt.year.unique())}')
print()

# Distribution des résultats sur l'ensemble des CdM
wc['result'] = wc.apply(
    lambda r: 'Domicile/Neutre' if r['home_score'] > r['away_score']
              else ('Nul' if r['home_score'] == r['away_score'] else 'Extérieur'),
    axis=1
)
print('Distribution des résultats (World Cup, phase finale) :')
print(wc['result'].value_counts(normalize=True).map('{:.1%}'.format).to_string())

In [ ]:
# Distribution des résultats sur l'ensemble du dataset (référence)
df['result'] = df.apply(
    lambda r: 'Home' if r['home_score'] > r['away_score']
              else ('Draw' if r['home_score'] == r['away_score'] else 'Away'),
    axis=1
)
print('Distribution globale (tous tournois) :')
print(df['result'].value_counts(normalize=True).map('{:.1%}'.format).to_string())
print()

# Nombre de matchs par décennie
df['decade'] = (df['date'].dt.year // 10) * 10
print('Matchs par décennie :')
print(df.groupby('decade').size().to_string())

## 5. Export — elo_history prêt pour le feature engineering

In [ ]:
# Sauvegarde de elo_history pour réutilisation dans features.py et train.py
elo_history.to_csv('../../ml/data/elo_history.csv', index=False)
elo_latest.to_csv('../../ml/data/elo_latest.csv', index=False)

print(f'elo_history.csv sauvegardé — {len(elo_history):,} lignes')
print(f'elo_latest.csv  sauvegardé — {len(elo_latest):,} lignes (classement actuel)')